In [1]:
import torch

def main():
    print("="*60)
    print(" [실험 4] 크로스 어텐션 주체성: 주객전도 시 발생하는 정보 소실율 검증")
    print("="*60)

    torch.manual_seed(101)
    d_model = 128
    len_enc = 16  # 인코더: 긴 영어 원문 context
    len_dec = 4   # 디코더: 현재 짧게 생성된 한국어 토큰

    # 인코더는 명확한 정보 구획을 가짐 (특정 타겟 정보가 포함됨)
    Encoder_Outputs = torch.randn(1, len_enc, d_model)
    # 디코더는 특정 단서(조사, 어미 등)를 타겟팅하는 예리한 상태
    Decoder_Outputs = torch.randn(1, len_dec, d_model) * 2.0

    # ----------------------------------------------------
    # Case A: 정석 구조 (Q = 디코더, K/V = 인코더)
    # ----------------------------------------------------
    Q_A = Decoder_Outputs
    K_A = Encoder_Outputs
    attn_scores_A = torch.matmul(Q_A, K_A.transpose(-2, -1)) / (d_model ** 0.5)
    attn_prob_A = torch.softmax(attn_scores_A, dim=-1)

    # 집중도(Sparsity) 분석: 균등 분포 대비 얼마나 뾰족하게 정보를 타겟팅하는가?
    uniform_ent = -torch.log(torch.tensor(1.0 / len_enc))
    ent_A = -torch.sum(attn_prob_A * torch.log(attn_prob_A + 1e-9), dim=-1).mean()
    info_gain_A = (uniform_ent - ent_A).item()

    # ----------------------------------------------------
    # Case B: 주객전도 역구조 (Q = 인코더, K/V = 디코더)
    # ----------------------------------------------------
    Q_B = Encoder_Outputs
    K_B = Decoder_Outputs
    attn_scores_B = torch.matmul(Q_B, K_B.transpose(-2, -1)) / (d_model ** 0.5)
    attn_prob_B = torch.softmax(attn_scores_B, dim=-1)

    ent_B = -torch.sum(attn_prob_B * torch.log(attn_prob_B + 1e-9), dim=-1).mean()
    uniform_ent_B = -torch.log(torch.tensor(1.0 / len_dec))
    info_gain_B = (uniform_ent_B - ent_B).item()

    print("정보이론적 효율성 스코어 대조")
    print(f" [정석 설계] 디코더가 질문(Q)일 때 맥락 압축 이득 (Information Gain): {info_gain_A:.4f}")
    print(f" [주객전도] 인코더가 질문(Q)일 때 맥락 압축 이득 (Information Gain): {info_gain_B:.4f}")
    print("-" * 60)

    # 텍스트 그래프로 시각화
    print("어텐션 에너지 분산 스펙트럼 (길수록 타겟팅이 정확함)")
    print(f" 디코더가 Q: [{'█' * int(info_gain_A * 10)}{'░' * (20 - int(info_gain_A * 10))}] (선택적 필터링 작동)")
    print(f" 인코더가 Q: [{'█' * int(info_gain_B * 10)}{'░' * (20 - int(info_gain_B * 10))}] (노이즈 범람 및 공간 오염)")

    print("\n 결론:")
    print(" -> 디코더가 질문을 던져야 인코더의 방대한 데이터 중 '지금 쓸 것'만 압축해서 가져옵니다.")
    print(" -> 반대로 인코더가 질문을 던지면 디코더의 빈약한 생성 공간에 모든 인코더 토큰이")
    print("    서로 밀고 들어와 정보 과부하 및 맥락 오염(Dissipation)을 유발합니다.")
    print("="*60)

if __name__ == "__main__":
    main()

 [실험 4] 크로스 어텐션 주체성: 주객전도 시 발생하는 정보 소실율 검증
정보이론적 효율성 스코어 대조
 [정석 설계] 디코더가 질문(Q)일 때 맥락 압축 이득 (Information Gain): 1.1074
 [주객전도] 인코더가 질문(Q)일 때 맥락 압축 이득 (Information Gain): 0.6502
------------------------------------------------------------
어텐션 에너지 분산 스펙트럼 (길수록 타겟팅이 정확함)
 디코더가 Q: [███████████░░░░░░░░░] (선택적 필터링 작동)
 인코더가 Q: [██████░░░░░░░░░░░░░░] (노이즈 범람 및 공간 오염)

 결론:
 -> 디코더가 질문을 던져야 인코더의 방대한 데이터 중 '지금 쓸 것'만 압축해서 가져옵니다.
 -> 반대로 인코더가 질문을 던지면 디코더의 빈약한 생성 공간에 모든 인코더 토큰이
    서로 밀고 들어와 정보 과부하 및 맥락 오염(Dissipation)을 유발합니다.
